In [2]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

# Set the path to the file you'd like to load
file_path = "D202.csv"

# Load the latest version
df = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "jaganadhg/house-hold-energy-data",
  file_path,
  # Provide any additional arguments like
  # sql_query or pandas_kwargs. See the
  # documenation for more information:
  # https://github.com/Kaggle/kagglehub/blob/main/README.md#kaggledatasetadapterpandas
)

print("First 5 records:", df.head())

/tmp/ipykernel_2034/2337708391.py:8: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df = kagglehub.load_dataset(


100%|██████████| 267k/267k [00:00<00:00, 441kB/s]

Extracting zip of D202.csv...


First 5 records:              TYPE        DATE START TIME END TIME  USAGE UNITS    COST  NOTES
0  Electric usage  10/22/2016       0:00     0:14   0.01   kWh  $0.00     NaN
1  Electric usage  10/22/2016       0:15     0:29   0.01   kWh  $0.00     NaN
2  Electric usage  10/22/2016       0:30     0:44   0.01   kWh  $0.00     NaN
3  Electric usage  10/22/2016       0:45     0:59   0.01   kWh  $0.00     NaN
4  Electric usage  10/22/2016       1:00     1:14   0.01   kWh  $0.00     NaN


In [3]:
import pandas as pd

def aggregate_to_hourly(df: pd.DataFrame) -> pd.DataFrame:
    """
    Convert smart meter data with separate DATE and START TIME columns into
    hourly aggregated usage.

    Expected columns:
        - 'DATE' (e.g. '10/22/2016')
        - 'START TIME' (e.g. '0:00')
        - 'USAGE' (numeric, kWh)

    Returns:
        DataFrame with columns:
        - 'timestamp' (hourly timestamp)
        - 'usage_kWh' (sum of usage within the hour)
    """
    # 1. Combine DATE + START TIME into single datetime
    df['timestamp'] = pd.to_datetime(df['DATE'] + ' ' + df['START TIME'])

    # 2. Round or floor to the hour (e.g. 00:15 → 00:00)
    df['hour'] = df['timestamp'].dt.floor('H')

    # 3. Convert USAGE to numeric (in case of strings)
    df['USAGE'] = pd.to_numeric(df['USAGE'], errors='coerce')

    # 4. Group by the hourly timestamp and sum usage
    hourly = (
        df.groupby('hour', as_index=False)['USAGE']
          .sum()
          .rename(columns={'hour': 'timestamp', 'USAGE': 'usage_kWh'})
    )

    # Sort by datetime just to be neat
    hourly = hourly.sort_values('timestamp').reset_index(drop=True)

    return hourly


In [4]:
hourly_df = aggregate_to_hourly(df)

/tmp/ipykernel_2034/2543286571.py:22: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df['hour'] = df['timestamp'].dt.floor('H')


In [5]:
hourly_df

,timestamp,usage_kWh
0,2016-10-22 00:00:00,0.04
1,2016-10-22 01:00:00,0.04
2,2016-10-22 02:00:00,0.08
3,2016-10-22 03:00:00,0.04
4,2016-10-22 04:00:00,0.04
...,...,...
17585,2018-10-24 19:00:00,0.12
17586,2018-10-24 20:00:00,0.12
17587,2018-10-24 21:00:00,0.12
17588,2018-10-24 22:00:00,0.08


In [6]:
hourly_df.describe()

,timestamp,usage_kWh
count,17590,17590.000000
mean,2017-10-23 11:37:06.310403584,0.487818
min,2016-10-22 00:00:00,0.000000
25%,2017-04-23 06:15:00,0.120000
50%,2017-10-23 11:30:00,0.200000
75%,2018-04-24 17:45:00,0.480000
max,2018-10-24 23:00:00,9.440000
std,NaN,0.842079


In [7]:
hourly_df['day'] = hourly_df['timestamp'].dt.day
hourly_df['day_name'] = hourly_df['timestamp'].dt.day_name()
hourly_df['month'] = hourly_df['timestamp'].dt.month
hourly_df['day_of_week'] = hourly_df['timestamp'].dt.dayofweek
hourly_df['day_of_year'] = hourly_df['timestamp'].dt.dayofyear
hourly_df['week_of_year'] = hourly_df['timestamp'].dt.isocalendar().week
hourly_df['hour'] = hourly_df['timestamp'].dt.hour
hourly_df['is_weekend'] = hourly_df['day_name'].isin(['Saturday','Sunday']).astype(int)
hourly_df['usage_change'] = hourly_df['usage_kWh'].diff().fillna(0)
hourly_df['rolling_mean_3'] = hourly_df['usage_kWh'].rolling(3).mean().fillna(method='bfill')
hourly_df['rolling_std_3'] = hourly_df['usage_kWh'].rolling(3).std().fillna(method='bfill')
hourly_df['usage_diff_3h'] = hourly_df['usage_kWh'].diff(3).fillna(0)


/tmp/ipykernel_2034/2020092959.py:10: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  hourly_df['rolling_mean_3'] = hourly_df['usage_kWh'].rolling(3).mean().fillna(method='bfill')
/tmp/ipykernel_2034/2020092959.py:11: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  hourly_df['rolling_std_3'] = hourly_df['usage_kWh'].rolling(3).std().fillna(method='bfill')


In [8]:
hourly_df.head()

,timestamp,usage_kWh,day,day_name,month,day_of_week,day_of_year,week_of_year,hour,is_weekend,usage_change,rolling_mean_3,rolling_std_3,usage_diff_3h
0,2016-10-22 00:00:00,0.04,22,Saturday,10,5,296,42,0,1,0.00,0.053333,0.023094,0.0
1,2016-10-22 01:00:00,0.04,22,Saturday,10,5,296,42,1,1,0.00,0.053333,0.023094,0.0
2,2016-10-22 02:00:00,0.08,22,Saturday,10,5,296,42,2,1,0.04,0.053333,0.023094,0.0
3,2016-10-22 03:00:00,0.04,22,Saturday,10,5,296,42,3,1,-0.04,0.053333,0.023094,0.0
4,2016-10-22 04:00:00,0.04,22,Saturday,10,5,296,42,4,1,0.00,0.053333,0.023094,0.0


In [9]:
FEATURE_COLS = [
    'usage_kWh','day','day_of_year','week_of_year','hour','day_of_week',
    'is_weekend','usage_change','rolling_mean_3','rolling_std_3','usage_diff_3h','month'
]

In [10]:
data=hourly_df[FEATURE_COLS]

In [11]:
data.head()

,usage_kWh,day,day_of_year,week_of_year,hour,day_of_week,is_weekend,usage_change,rolling_mean_3,rolling_std_3,usage_diff_3h,month
0,0.04,22,296,42,0,5,1,0.00,0.053333,0.023094,0.0,10
1,0.04,22,296,42,1,5,1,0.00,0.053333,0.023094,0.0,10
2,0.08,22,296,42,2,5,1,0.04,0.053333,0.023094,0.0,10
3,0.04,22,296,42,3,5,1,-0.04,0.053333,0.023094,0.0,10
4,0.04,22,296,42,4,5,1,0.00,0.053333,0.023094,0.0,10


In [12]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

#final

In [13]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout


mid = len(data) // 2
first_half = data.iloc[:mid].copy()
second_half = data.iloc[mid:].copy()

In [14]:
import os
import math
import numpy as np
import pandas as pd
from tqdm import tqdm
from datetime import timedelta
import joblib
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

# -------------------------------------
# CONFIG
# -------------------------------------
SEQ_LEN = 168          # 1 week history input
PRED_HORIZON = 168       # 1 week forecast
BATCH_SIZE = 64
EPOCHS_INITIAL =10
EPOCHS_FINETUNE = 5
FINE_TUNE_EVERY = 7        # fine-tune every 1 predicted week
MODEL_DIR = "./models"
os.makedirs(MODEL_DIR, exist_ok=True)


# -------------------------------------
# FEATURE PREP
# -------------------------------------
FEATURES = [
    'usage_kWh',
    'day', 'day_of_year', 'week_of_year',
    'hour', 'day_of_week', 'is_weekend',
    'usage_change', 'rolling_mean_3', 'rolling_std_3',
    'usage_diff_3h', 'month'
]


def prepare_df(df):
    df = df[FEATURES]
    df = df.interpolate().fillna(method="bfill").fillna(method="ffill")
    return df


# CREATE WINDOWS
def create_sequences(data, target, seq_len=SEQ_LEN, pred_horizon=PRED_HORIZON):
    X, y = [], []
    total = len(data)
    for i in range(total - seq_len - pred_horizon):
        X.append(data[i:i + seq_len])
        y.append(target[i + seq_len:i + seq_len + pred_horizon])
    return np.array(X), np.array(y)


def build_model(n_features, seq_len=SEQ_LEN, pred_horizon=PRED_HORIZON):
    model = models.Sequential([
        layers.Input(shape=(seq_len, n_features)),
        layers.LSTM(128, return_sequences=True),
        layers.Dropout(0.2),
        layers.LSTM(64),
        layers.Dropout(0.2),
        layers.Dense(pred_horizon)
    ])
    model.compile(optimizer='adam', loss='mse')
    return model


# TRAIN ON YEAR1
def train_initial_model(df1):
    scaler = MinMaxScaler()
    scaled = scaler.fit_transform(df1)

    X, y = create_sequences(
        scaled,
        scaled[:, 0],  # target
        SEQ_LEN,
        PRED_HORIZON
    )

    model = build_model(n_features=scaled.shape[1])
    model.fit(
        X, y,
        epochs=EPOCHS_INITIAL,
        batch_size=BATCH_SIZE,
        validation_split=0.1,
        callbacks=[callbacks.EarlyStopping(patience=5, restore_best_weights=True)],
        verbose=2
    )


    # Save model
    model.save(f"{MODEL_DIR}/initial_model.keras")
    # Save scaler
    joblib.dump(scaler, f"{MODEL_DIR}/scaler.save")
    return model, scaler


# PREDICT ON YEAR2
def weekly_predict_and_finetune(model, scaler, df1, df2):

    combined = pd.concat([df1, df2])
    combined = combined.reset_index(drop=True)

    scaled = scaler.transform(combined)

    results = []

    total_len = len(df1) + len(df2)

    # start predicting after df1
    start_idx = len(df1)

    num_weeks = (len(df2) - PRED_HORIZON) // PRED_HORIZON

    week_index = 0

    for w in range(num_weeks):

        week_start = start_idx + w * PRED_HORIZON
        seq_start = week_start - SEQ_LEN
        seq_end = week_start

        if seq_start < 0:
            continue

        # Input sequence
        x = scaled[seq_start:seq_end].reshape(1, SEQ_LEN, -1)

        pred_scaled = model.predict(x, verbose=0)[0]

        reconstructed = np.zeros((PRED_HORIZON, scaled.shape[1]))
        reconstructed[:, 0] = pred_scaled

        pred = scaler.inverse_transform(reconstructed)[:, 0]

        true_vals = combined.iloc[week_start:week_start + PRED_HORIZON]['usage_kWh'].values

        # Metrics
        mae = mean_absolute_error(true_vals, pred)
        rmse = math.sqrt(mean_squared_error(true_vals, pred))

        results.append({
            "week_number": w + 1,
            "start_index": week_start,
            "MAE": mae,
            "RMSE": rmse
        })

        week_index += 1

        # Monthly fine-tune
        if (week_index % FINE_TUNE_EVERY) == 0:

            ft_start = max(0, seq_end - (SEQ_LEN + 4 * PRED_HORIZON))
            ft_end = seq_end + PRED_HORIZON

            ft_data = scaled[ft_start:ft_end]

            X_ft, Y_ft = create_sequences(
                ft_data,
                ft_data[:, 0],
                SEQ_LEN,
                PRED_HORIZON
            )

            if len(X_ft) > 0:
                model.fit(X_ft, Y_ft, epochs=EPOCHS_FINETUNE, batch_size=32, verbose=0)
                model.save(f"{MODEL_DIR}/model_week_{week_index}.keras")

    return pd.DataFrame(results)



def run_lstm_workflow(df_year1, df_year2):

    df1 = prepare_df(df_year1)
    df2 = prepare_df(df_year2)

    print("\n Training initial model on YEAR 1...")
    model, scaler = train_initial_model(df1)

    print("\n Predicting weekly on YEAR 2 with fine-tuning...")
    results = weekly_predict_and_finetune(model, scaler, df1, df2)

    print("\nDone!")
    return results


In [15]:
    #weekly
    df1 = prepare_df(first_half)
    df2 = prepare_df(second_half)

    print("\n Training initial model on YEAR 1...")
    model, scaler = train_initial_model(df1)



/tmp/ipykernel_2034/758234969.py:41: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df = df.interpolate().fillna(method="bfill").fillna(method="ffill")
/tmp/ipykernel_2034/758234969.py:41: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df = df.interpolate().fillna(method="bfill").fillna(method="ffill")



 Training initial model on YEAR 1...
Epoch 1/10
119/119 - 7s - 59ms/step - loss: 0.0104 - val_loss: 0.0012
Epoch 2/10
119/119 - 2s - 15ms/step - loss: 0.0097 - val_loss: 0.0017
Epoch 3/10
119/119 - 2s - 15ms/step - loss: 0.0096 - val_loss: 0.0012
Epoch 4/10
119/119 - 2s - 16ms/step - loss: 0.0095 - val_loss: 0.0015
Epoch 5/10
119/119 - 2s - 15ms/step - loss: 0.0095 - val_loss: 0.0014
Epoch 6/10
119/119 - 2s - 20ms/step - loss: 0.0094 - val_loss: 0.0012
Epoch 7/10
119/119 - 2s - 18ms/step - loss: 0.0093 - val_loss: 0.0012
Epoch 8/10
119/119 - 2s - 16ms/step - loss: 0.0090 - val_loss: 0.0012
Epoch 9/10
119/119 - 2s - 15ms/step - loss: 0.0089 - val_loss: 0.0012
Epoch 10/10
119/119 - 3s - 23ms/step - loss: 0.0089 - val_loss: 0.0012


In [16]:
from tensorflow.keras.models import load_model
import joblib

# Load model
model = load_model(f"{MODEL_DIR}/initial_model.keras")

# Load scaler
scaler = joblib.load(f"{MODEL_DIR}/scaler.save")

In [17]:
print("\n Predicting weekly on YEAR 2 with fine-tuning...")
results = weekly_predict_and_finetune(model, scaler, df1, df2)

print("\nDone!")



 Predicting weekly on YEAR 2 with fine-tuning...

Done!


In [18]:
print(results)

    week_number  start_index       MAE      RMSE
0             1         8795  0.331127  0.408048
1             2         8963  0.534331  0.888271
2             3         9131  0.629989  1.377255
3             4         9299  0.659665  1.292438
4             5         9467  0.503792  0.533821
5             6         9635  0.799872  1.325934
6             7         9803  0.972990  1.788461
7             8         9971  0.726420  1.371210
8             9        10139  0.739749  1.380085
9            10        10307  0.843360  1.627686
10           11        10475  0.490380  0.638114
11           12        10643  0.621325  1.248945
12           13        10811  0.611813  1.243888
13           14        10979  0.729540  1.445693
14           15        11147  0.750044  1.098034
15           16        11315  0.402187  0.485436
16           17        11483  0.575570  1.023157
17           18        11651  0.836240  1.530290
18           19        11819  0.788513  1.333128
19           20     

In [19]:
results.describe()

,week_number,start_index,MAE,RMSE
count,51.000000,51.00000,51.000000,51.000000
mean,26.000000,12995.00000,0.456054,0.735838
std,14.866069,2497.49955,0.207277,0.436801
min,1.000000,8795.00000,0.183117,0.259557
25%,13.500000,10895.00000,0.288054,0.405766
50%,26.000000,12995.00000,0.389979,0.528121
75%,38.500000,15095.00000,0.616569,1.111946
max,51.000000,17195.00000,0.972990,1.788461


In [20]:
results.to_csv('results.csv', index=False)